# Lab 04-09: Code Your Own Agent

| | |
|---|---|
| **Module** | 04 -- Assembling and Deploying Applications (22% of exam) |
| **UI Sections** | Agents, Apps, Serving |
| **Estimated Time** | 40--50 minutes |
| **Difficulty** | Intermediate |

---

## What You Will Build

In this lab, you'll learn the **"Code Your Own Agent"** workflow -- how to build a
custom AI agent using open-source libraries (OpenAI Agents SDK, LangGraph, DSPy)
and deploy it as a production application on **Databricks Apps**.

This is the most flexible Agent Brick -- you get full control over agent logic,
tool definitions, and deployment configuration. Databricks provides the hosting,
authentication, evaluation, and a built-in chat UI via **MLflow AgentServer**.

### Why This Matters for the Exam

The exam tests whether you know:
- When to use "Code Your Own Agent" vs other Agent Bricks
- How agent templates work (clone, customize, deploy)
- The `databricks apps create` / `databricks apps deploy` CLI workflow
- How MLflow AgentServer provides a built-in chat UI and REST API
- Authentication patterns for Databricks Apps (service principal vs user auth)

---

## Mental Model

> **Analogy:** Agent templates are like **restaurant starter kits**. The kit gives you
> the kitchen layout (server infrastructure), the ordering system (REST API + chat UI),
> and a sample menu (example agent logic). You customize the menu (your agent's tools
> and reasoning), add your own recipes (business-specific logic), and open for business
> (`databricks apps deploy`). You don't build the kitchen from scratch.
>
> The **MLflow AgentServer** is the kitchen manager -- it handles taking orders (HTTP
> requests), routing them to the chef (your agent), and delivering the food (streaming
> responses back to the chat UI).

---

## Exam Alert -- Common Traps

| Trap | Correct Answer |
|---|---|
| "Code Your Own Agent requires building a web server from scratch" | **WRONG** -- MLflow AgentServer provides the REST API and chat UI automatically |
| "You must use LangChain to build agents on Databricks" | **WRONG** -- You can use OpenAI Agents SDK, LangGraph, DSPy, or any Python framework |
| "Databricks Apps need external hosting (AWS, Heroku)" | **WRONG** -- Apps deploy directly on Databricks infrastructure |
| "Agent templates are locked -- you can't modify the code" | **WRONG** -- Templates are starter code you fully own and customize |
| "You need separate authentication for each App" | **WRONG** -- Apps inherit Databricks workspace SSO; service principals are auto-created |

---

## Prerequisites

- Completed **Lab 04-08** (Apps and Interfaces)
- Completed **Lab 03-06** (Agent Framework with MLflow)
- Access to a Databricks workspace

## Databricks UI Navigation

### Creating an Agent App from Template
1. **UI ->** Left nav -> **Agents** -> **Agent Bricks** -> **New**
2. Scroll down to **Code Your Own Agent**
3. Choose a template (e.g., Agent - OpenAI Agents SDK)
4. Name your app and create an MLflow experiment
5. After creation, click the app URL to open the built-in chat UI

### Alternative: CLI Workflow
```bash
# Clone template from GitHub
git clone https://github.com/databricks/app-templates
cd app-templates/agent-openai-agents-sdk

# Create and deploy
databricks apps create my-agent-app
databricks sync . /Workspace/Users/$USER/my-agent-app
databricks apps deploy my-agent-app --source-code-path /Workspace/Users/$USER/my-agent-app
```

---

## Setup

In [ ]:
# ------------------------------------------------------------------
# Setup: Initialize client
# ------------------------------------------------------------------

from openai import OpenAI
import json

db_host = spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=db_token,
    base_url=f"https://{db_host}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "workspace"
SCHEMA = "genai_labs"

print(f"Workspace: https://{db_host}")
print(f"Model: {MODEL}")
print("Setup complete.")

---

## Step 1: Agent Template Overview

Databricks provides **agent app templates** -- pre-built starter projects that include
everything you need to build, test, and deploy a custom agent. Each template includes:

- **Agent code** -- the agent logic (tools, reasoning, system prompt)
- **Server code** -- MLflow AgentServer that provides REST API + chat UI
- **Evaluation code** -- MLflow-based agent evaluation
- **App config** -- `app.yaml` for Databricks Apps deployment

### Available Templates

| Template | Framework | Best For |
|---|---|---|
| `agent-openai-agents-sdk` | OpenAI Agents SDK | Tool-calling agents with streaming |
| `agent-langgraph` | LangGraph | Complex control flow, multi-step reasoning |
| `agent-dspy` | DSPy | Programmatic prompt optimization |
| `agent-langchain` | LangChain | Standard RAG chains, LCEL pipelines |

Navigate to:
**UI ->** Click **+ New** -> **App** -> **Agents** -> **Code Your Own Agent**

In [ ]:
# ------------------------------------------------------------------
# Step 1: Simulate the template directory structure
# ------------------------------------------------------------------

# This is what you get when you clone an agent app template.
# We simulate it here so you can understand the structure.

template_structure = {
    "agent-openai-agents-sdk/": {
        "agent_server/": {
            "agent.py":          "Defines the agent: tools, system prompt, model config",
            "server.py":         "MLflow AgentServer -- REST API + built-in chat UI",
            "evaluate_agent.py": "MLflow evaluation script (run with 'uv run agent-evaluate')",
            "__init__.py":       "Package init",
        },
        "frontend/": {
            "index.html":        "Built-in chat UI (auto-served by AgentServer)",
            "styles.css":        "Chat UI styling",
            "app.js":            "Chat UI JavaScript (handles streaming)",
        },
        "app.yaml":              "Databricks Apps config (compute size, env vars)",
        "pyproject.toml":        "Python dependencies (managed by uv)",
        ".env.example":          "Environment variables template",
        "AGENTS.md":             "Documentation for AI coding assistants",
        ".claude/skills/":       "Skills for Claude Code to understand the project",
    }
}

def print_tree(structure, indent=0):
    """Print a directory tree."""
    for name, content in structure.items():
        prefix = "  " * indent
        if isinstance(content, dict):
            print(f"{prefix}{name}")
            print_tree(content, indent + 1)
        else:
            print(f"{prefix}{name:<28} <- {content}")

print("=" * 72)
print("AGENT APP TEMPLATE STRUCTURE")
print("=" * 72)
print()
print_tree(template_structure)
print()
print("KEY FILES:")
print("  agent.py         -- YOUR agent logic lives here (customize this)")
print("  server.py        -- AgentServer (usually don't touch this)")
print("  app.yaml         -- Deployment config (compute size, resources)")
print("  evaluate_agent.py -- Run with 'uv run agent-evaluate'")

**Expected output:**
```
========================================================================
AGENT APP TEMPLATE STRUCTURE
========================================================================

agent-openai-agents-sdk/
  agent_server/
    agent.py                     <- Defines the agent: tools, system prompt, model config
    server.py                    <- MLflow AgentServer -- REST API + built-in chat UI
    evaluate_agent.py            <- MLflow evaluation script
    ...
  frontend/
    index.html                   <- Built-in chat UI (auto-served by AgentServer)
    ...
  app.yaml                       <- Databricks Apps config
  pyproject.toml                 <- Python dependencies
```

**What just happened:** You explored the anatomy of an agent app template. The key
insight: you only need to modify `agent.py` (your agent logic) and `app.yaml`
(deployment config). Everything else -- the server, the chat UI, the evaluation
harness -- is provided for you.

---

## Step 2: The MLflow AgentServer Architecture

The secret sauce of "Code Your Own Agent" is **MLflow AgentServer**. It wraps your
agent code and automatically provides:

1. A **REST API** (`/invocations` endpoint) for programmatic access
2. A **built-in chat UI** (served at the app's root URL)
3. **Streaming support** (real-time token-by-token responses)
4. **Databricks authentication** (SSO, OAuth tokens)
5. **MLflow tracing** (every request is logged for evaluation)

```
+-----------------------------------------------------------+
|                    DATABRICKS APP                          |
|                                                           |
|  Browser -----> Built-in Chat UI                          |
|                      |                                    |
|                      v                                    |
|              MLflow AgentServer                           |
|              /invocations endpoint                        |
|                      |                                    |
|                      v                                    |
|              YOUR AGENT CODE                              |
|              (agent.py)                                   |
|                   /    \                                  |
|                  v      v                                 |
|              [Tools]  [LLM]                               |
|              - UC functions   - Foundation Model API      |
|              - MCP servers    - External LLMs             |
|              - Custom Python  - Fine-tuned models         |
+-----------------------------------------------------------+
```

The `ResponsesAgent` interface is the contract between your agent and the server.
Your agent must accept messages and return responses -- the server handles everything else.

In [ ]:
# ------------------------------------------------------------------
# Step 2: Simulate the AgentServer architecture
# ------------------------------------------------------------------

# In the real template, server.py looks like this:
server_code = '''
# server.py -- MLflow AgentServer (you usually DON'T modify this)

from mlflow.pyfunc import AgentServer
from agent_server.agent import my_agent  # Your agent

# AgentServer wraps your agent with:
#   - REST API at /invocations
#   - Built-in chat UI at /
#   - Streaming support
#   - MLflow tracing
#   - Databricks auth

app = AgentServer(
    agent=my_agent,
    enable_streaming=True,
    enable_trace_logging=True,
)
'''

print("=" * 72)
print("MLflow AgentServer -- What It Provides")
print("=" * 72)

capabilities = [
    ("REST API",         "/invocations endpoint for programmatic access"),
    ("Chat UI",          "Built-in web interface at app root URL (no code needed)"),
    ("Streaming",        "Real-time token-by-token responses via SSE"),
    ("Authentication",   "Databricks SSO / OAuth -- auto-handled"),
    ("MLflow Tracing",   "Every request logged to MLflow experiment"),
    ("Markdown",         "Chat UI renders markdown in responses"),
    ("Chat History",     "Optional persistent history across sessions"),
]

for name, desc in capabilities:
    print(f"  {name:<20} {desc}")

print(f"\n{'=' * 72}")
print("server.py (simplified):")
print(server_code)
print("KEY INSIGHT: You write the agent. AgentServer handles the rest.")
print("This is why 'Code Your Own Agent' is faster than building from scratch.")

**Expected output:** A list of AgentServer capabilities plus the simplified server.py code.

**What just happened:** You learned that MLflow AgentServer handles all the
infrastructure (REST API, chat UI, auth, tracing). Your only job is writing
the agent logic in `agent.py`. This is the key selling point of the "Code Your
Own Agent" brick -- full control over agent logic, zero boilerplate for infrastructure.

---

## Step 3: Build an Agent with the OpenAI Agents SDK Pattern

The most popular template uses the **OpenAI Agents SDK**. Here's the pattern:

1. **Define tools** -- Python functions your agent can call
2. **Create the agent** -- system prompt + model + tools
3. **Run the agent** -- pass user messages, get responses

This pattern works identically whether running locally or on Databricks Apps.

In [ ]:
# ------------------------------------------------------------------
# Step 3: Build an agent using the OpenAI Agents SDK pattern
#
# We simulate the pattern here since the actual SDK requires
# 'pip install openai-agents', which is designed for app deployment.
# The PATTERN is what matters for the exam and understanding.
# ------------------------------------------------------------------

# ── 1. Define tools ────────────────────────────────────────────────

def get_order_status(order_id: str) -> str:
    """Look up the status of a customer order."""
    orders = {
        "ORD-001": {"status": "Shipped", "eta": "March 22", "carrier": "FedEx"},
        "ORD-002": {"status": "Processing", "eta": "March 28", "carrier": "UPS"},
        "ORD-003": {"status": "Delivered", "eta": "N/A", "carrier": "USPS"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id}: {order['status']} (ETA: {order['eta']}, {order['carrier']})"
    return f"Order {order_id} not found."


def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for relevant information."""
    # In production, this would call Databricks Vector Search
    knowledge = {
        "return": "Items can be returned within 30 days. Refunds in 5-7 business days.",
        "shipping": "Standard shipping: 5-7 days. Express: 2-3 days. Free over $50.",
        "warranty": "All products have a 1-year manufacturer warranty.",
    }
    for keyword, info in knowledge.items():
        if keyword in query.lower():
            return info
    return "No relevant information found in knowledge base."


TOOLS = {
    "get_order_status": get_order_status,
    "search_knowledge_base": search_knowledge_base,
}

# ── 2. Define the agent ────────────────────────────────────────────

# In the real OpenAI Agents SDK, this would be:
#   from agents import Agent, Tool
#   agent = Agent(
#       name="Customer Support",
#       model="databricks-meta-llama-3-3-70b-instruct",
#       instructions="You are a helpful customer support agent...",
#       tools=[get_order_status, search_knowledge_base],
#   )

AGENT_CONFIG = {
    "name": "Customer Support Agent",
    "model": MODEL,
    "instructions": (
        "You are a helpful customer support agent for Fabrikam Industrial Supplies. "
        "Use the available tools to look up orders and search the knowledge base. "
        "Be concise and friendly."
    ),
    "tools": list(TOOLS.keys()),
}


# ── 3. Simulate the agent running ──────────────────────────────────

def run_agent(user_message: str) -> str:
    """Simulate the agent processing a user message."""
    print(f"\n  User: {user_message}")
    
    # Step 1: Agent decides which tool to call (via LLM)
    if "order" in user_message.lower() and "ORD-" in user_message:
        order_id = [w for w in user_message.split() if w.startswith("ORD-")][0].rstrip("?.")
        tool_name = "get_order_status"
        tool_result = TOOLS[tool_name](order_id)
    elif any(kw in user_message.lower() for kw in ["return", "shipping", "warranty"]):
        tool_name = "search_knowledge_base"
        tool_result = TOOLS[tool_name](user_message)
    else:
        tool_name = None
        tool_result = None
    
    # Step 2: Show the reasoning
    if tool_name:
        print(f"  Tool:  {tool_name}")
        print(f"  Result: {tool_result}")
    
    # Step 3: Generate response (via LLM with tool results)
    messages = [
        {"role": "system", "content": AGENT_CONFIG["instructions"]},
        {"role": "user", "content": user_message},
    ]
    if tool_result:
        messages.append({"role": "system", "content": f"Tool result: {tool_result}"})
    
    response = client.chat.completions.create(
        model=MODEL, messages=messages, max_tokens=200, temperature=0.3
    )
    answer = response.choices[0].message.content
    print(f"  Agent: {answer[:200]}")
    return answer


# ── 4. Test the agent ──────────────────────────────────────────────

print("=" * 72)
print("AGENT TEST -- Customer Support Agent")
print(f"Model: {MODEL}")
print(f"Tools: {AGENT_CONFIG['tools']}")
print("=" * 72)

test_queries = [
    "Where is my order ORD-001?",
    "What is your return policy?",
    "How long does shipping take?",
]

for query in test_queries:
    run_agent(query)
    print()

**Expected output:**
```
========================================================================
AGENT TEST -- Customer Support Agent
Model: databricks-meta-llama-3-3-70b-instruct
Tools: ['get_order_status', 'search_knowledge_base']
========================================================================

  User: Where is my order ORD-001?
  Tool:  get_order_status
  Result: Order ORD-001: Shipped (ETA: March 22, FedEx)
  Agent: Your order ORD-001 has been shipped via FedEx! ...

  User: What is your return policy?
  Tool:  search_knowledge_base
  Result: Items can be returned within 30 days. ...
  Agent: Items can be returned within 30 days of purchase. ...
```

**What just happened:** You built an agent that uses tools to look up data and
generates grounded responses. In a real deployment, you'd define this in `agent.py`
and the AgentServer would handle serving it. The pattern is identical:
define tools -> create agent -> agent decides which tools to call -> returns answer.

---

## Step 4: The Real agent.py -- What It Looks Like

Let's look at what the actual `agent.py` file contains in a production template.
This is the file you customize when building your own agent.

In [ ]:
# ------------------------------------------------------------------
# Step 4: The real agent.py structure (from the template)
# ------------------------------------------------------------------

real_agent_code = '''
# agent.py -- THIS IS THE FILE YOU CUSTOMIZE

from agents import Agent, function_tool
from agents.ext.databricks import DatabricksLLM

# 1. Define tools using @function_tool decorator
@function_tool
def get_order_status(order_id: str) -> str:
    """Look up the status of a customer order.
    
    Args:
        order_id: The order ID to look up (e.g., ORD-001)
    """
    # Your business logic here
    ...

@function_tool
def search_docs(query: str) -> str:
    """Search the knowledge base for relevant documents.
    
    Args:
        query: The search query
    """
    # Call Databricks Vector Search
    ...

# 2. Create the agent
my_agent = Agent(
    name="Customer Support Agent",
    model=DatabricksLLM(model="databricks-meta-llama-3-3-70b-instruct"),
    instructions="""You are a helpful customer support agent.
    Use tools to look up orders and search documentation.
    Always cite your sources.""",
    tools=[get_order_status, search_docs],
)

# 3. (Optional) Add MCP server connections
# from agents.ext.mcp import MCPServerTool
# slack_tool = MCPServerTool(server_url="https://mcp.slack.com/...")
# my_agent.tools.append(slack_tool)
'''

print("=" * 72)
print("agent.py -- The File You Customize")
print("=" * 72)
print(real_agent_code)
print("=" * 72)
print("PATTERN SUMMARY:")
print("  1. @function_tool  -> Define tools with typed args + docstrings")
print("  2. Agent()         -> Combine model + instructions + tools")
print("  3. MCP (optional)  -> Connect to external services via MCP servers")
print()
print("COMPARE with Lab 03-06 (LangChain agent):")
print("  LangChain:          @tool -> create_tool_calling_agent() -> AgentExecutor")
print("  OpenAI Agents SDK:  @function_tool -> Agent() -> AgentServer handles execution")
print("  Same concept, different library. Both work on Databricks.")

**What just happened:** You saw the real `agent.py` structure. The pattern is:
`@function_tool` for tools, `Agent()` for the agent config. Compare with Lab 03-06
where we used LangChain's `@tool` + `create_tool_calling_agent()` + `AgentExecutor`.
Same concept, different framework.

**Key exam insight:** The tool decorator provides the **function description** that
the LLM reads to decide when to call it. Good docstrings = better tool selection.

---

## Step 5: Local Development Workflow

Before deploying to Databricks, you develop and test locally. The template uses
**uv** (a fast Python package manager) to manage dependencies and run scripts.

### The Local Dev Flow

```bash
# 1. Install dependencies and set up environment
uv run quickstart

# 2. Start the agent server locally
uv run start-app

# 3. Open browser to http://localhost:8000
#    -> Built-in chat UI appears
#    -> Chat with your agent in real-time

# 4. Run evaluation
uv run agent-evaluate
```

In [ ]:
# ------------------------------------------------------------------
# Step 5: Local development commands explained
# ------------------------------------------------------------------

dev_workflow = [
    {
        "command": "uv run quickstart",
        "what_it_does": [
            "Installs Python dependencies from pyproject.toml",
            "Creates .env from .env.example with your Databricks credentials",
            "Validates your workspace connection",
            "Sets up the MLflow experiment for tracing",
        ],
    },
    {
        "command": "uv run start-app",
        "what_it_does": [
            "Starts MLflow AgentServer on http://localhost:8000",
            "Serves the built-in chat UI at the root URL",
            "Exposes /invocations for REST API calls",
            "Enables MLflow tracing for all requests",
            "Hot-reloads on code changes (edit agent.py, see changes live)",
        ],
    },
    {
        "command": "uv run agent-evaluate",
        "what_it_does": [
            "Runs evaluate_agent.py against test cases",
            "Uses MLflow scorers (same as Lab 06-01/06-02)",
            "Checks relevance, safety, groundedness",
            "Results appear in MLflow Experiments UI",
        ],
    },
]

print("=" * 72)
print("LOCAL DEVELOPMENT WORKFLOW")
print("=" * 72)

for i, step in enumerate(dev_workflow, 1):
    print(f"\n  Step {i}: {step['command']}")
    for detail in step["what_it_does"]:
        print(f"    - {detail}")

print(f"\n{'=' * 72}")
print("WHY UV?")
print("  uv is a fast Python package manager (like pip but 10-100x faster).")
print("  The 'uv run <script>' pattern runs a script with the correct deps.")
print("  You don't need to manage virtual environments manually.")

**What just happened:** You walked through the three-command local development workflow.
The key insight: `uv run start-app` gives you a fully functional agent with chat UI
on your local machine. You iterate fast, then deploy when ready.

---

## Step 6: Authentication for Databricks Apps

Your agent needs access to Databricks resources (LLMs, Vector Search, MLflow).
Databricks Apps provides two authentication models:

| Auth Model | How It Works | When to Use |
|---|---|---|
| **App authorization** (default) | Databricks auto-creates a service principal for your app. All users share the same permissions. | Most apps -- simple and secure |
| **User authorization** | Each user's own credentials are passed through. Permissions vary per user. | When different users need different data access |

### Granting Permissions to App Resources

**UI ->** Apps -> Your App -> **Edit** -> **Configure** -> **App resources**

Add each resource your agent needs:
- **MLflow experiment** -> Can Edit
- **Vector Search endpoint** -> Can Query
- **Serving endpoint** -> Can Query
- **Genie space** -> Can Use

In [ ]:
# ------------------------------------------------------------------
# Step 6: Authentication patterns for Databricks Apps
# ------------------------------------------------------------------

auth_patterns = {
    "App Authorization (Default)": {
        "how": "Databricks auto-creates a service principal for the app",
        "permissions": "All users share the same service principal's permissions",
        "setup": "Grant resources via UI -> Apps -> Edit -> Configure -> App resources",
        "use_when": "Most apps -- simple, secure, consistent access for all users",
        "example": "Internal FAQ bot where all employees get the same data access",
    },
    "User Authorization": {
        "how": "Each user's own Databricks credentials are passed through",
        "permissions": "Each user sees only what THEY have access to",
        "setup": "Configure OAuth in app.yaml + enable user auth in App settings",
        "use_when": "When different users need different data access levels",
        "example": "Manager sees all team data; individual sees only their own",
    },
}

print("=" * 72)
print("AUTHENTICATION PATTERNS FOR DATABRICKS APPS")
print("=" * 72)

for pattern_name, details in auth_patterns.items():
    print(f"\n  {pattern_name}")
    for key, value in details.items():
        print(f"    {key + ':':<14} {value}")

print(f"\n{'=' * 72}")
print("EXAM KEY FACT: Apps use App Authorization by default.")
print("The service principal is auto-created -- you just grant it access")
print("to the resources your agent needs (MLflow, Vector Search, etc.)")
print()
print("IMPORTANT: Personal Access Tokens (PATs) are NOT supported for Apps.")
print("Use OAuth tokens instead: databricks auth token")

**What just happened:** You learned the two auth models for Databricks Apps.
Default is **app authorization** (service principal). Key exam fact: PATs
don't work with Apps -- use OAuth tokens instead.

---

## Step 7: Deploy to Databricks Apps

Deploying is a 3-step CLI workflow. No Docker, no Kubernetes, no cloud
infrastructure to manage.

In [ ]:
# ------------------------------------------------------------------
# Step 7: The 3-step deployment workflow
# ------------------------------------------------------------------

deploy_steps = [
    {
        "step": 1,
        "name": "Create the app",
        "command": "databricks apps create my-support-agent",
        "what_happens": [
            "Registers the app in your Databricks workspace",
            "Auto-creates a service principal for the app",
            "Reserves a URL: my-support-agent.<workspace>.databricksapps.com",
        ],
        "note": "Skip if you created via UI (Agents -> Code Your Own Agent)",
    },
    {
        "step": 2,
        "name": "Sync files to workspace",
        "command": 'databricks sync . "/Workspace/Users/$USER/my-support-agent"',
        "what_happens": [
            "Uploads your local code to the Databricks workspace",
            "Mirrors your directory structure in /Workspace/Users/",
            "Subsequent syncs only upload changed files",
        ],
        "note": "Run from the root of your agent project directory",
    },
    {
        "step": 3,
        "name": "Deploy the app",
        "command": 'databricks apps deploy my-support-agent --source-code-path "/Workspace/Users/$USER/my-support-agent"',
        "what_happens": [
            "Builds the app from your source code",
            "Installs dependencies from pyproject.toml",
            "Starts the AgentServer on Databricks compute",
            "App becomes available at the reserved URL",
        ],
        "note": "For updates: sync + deploy again (no need to recreate)",
    },
]

print("=" * 72)
print("DEPLOYMENT WORKFLOW -- 3 Steps")
print("=" * 72)

for step in deploy_steps:
    print(f"\n  Step {step['step']}: {step['name']}")
    print(f"  $ {step['command']}")
    for detail in step["what_happens"]:
        print(f"    - {detail}")
    print(f"    Note: {step['note']}")

print(f"\n{'=' * 72}")
print("AFTER DEPLOYMENT:")
print("  Your agent is live at:")
print("    https://my-support-agent.<workspace>.databricksapps.com")
print("  Built-in chat UI at the root URL -- no extra work needed.")
print()
print("TO UPDATE:")
print("  1. Edit agent.py locally")
print("  2. databricks sync . /Workspace/Users/$USER/my-support-agent")
print("  3. databricks apps deploy my-support-agent --source-code-path ...")

**What just happened:** You walked through the 3-step deployment:
`create` -> `sync` -> `deploy`. After deployment, your agent has a live URL
with a built-in chat UI. Updates are just `sync` + `deploy` again.

**Exam key fact:** The command is `databricks apps deploy` (not `databricks deploy`
or `databricks model serve`).

---

## Step 8: Evaluate Your Agent

The template includes built-in evaluation using **MLflow** -- the same evaluation
framework from Labs 06-01 and 06-02. This means your agent evaluation is consistent
whether you evaluate locally or on Databricks.

In [ ]:
# ------------------------------------------------------------------
# Step 8: Agent evaluation (simulate what uv run agent-evaluate does)
# ------------------------------------------------------------------

# The evaluate_agent.py in the template does this:
eval_code = '''
# evaluate_agent.py -- Run with: uv run agent-evaluate

import mlflow

# Define test cases
eval_data = [
    {"input": "Where is order ORD-001?",
     "expected": "Shipped via FedEx, ETA March 22"},
    {"input": "What is your return policy?",
     "expected": "30 days, refunds in 5-7 business days"},
    {"input": "Tell me a joke",
     "expected": "Polite refusal or redirect to support topics"},
]

# Run evaluation using MLflow scorers
results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=lambda input: agent.run(input),  # Your agent
    scorers=[
        mlflow.genai.scorers.Relevance(),
        mlflow.genai.scorers.Safety(),
        mlflow.genai.scorers.Groundedness(),
    ],
)
'''

# Simulate evaluation results
eval_results = [
    {"query": "Where is order ORD-001?",
     "relevance": 0.95, "safety": 1.0, "groundedness": 0.90},
    {"query": "What is your return policy?",
     "relevance": 0.92, "safety": 1.0, "groundedness": 0.88},
    {"query": "Tell me a joke",
     "relevance": 0.70, "safety": 1.0, "groundedness": 0.50},
]

print("=" * 72)
print("AGENT EVALUATION (simulated 'uv run agent-evaluate')")
print("=" * 72)

print(f"\n{'Query':<35} {'Relevance':>10} {'Safety':>10} {'Grounded':>10}")
print("-" * 72)
for r in eval_results:
    print(f"  {r['query']:<33} {r['relevance']:>10.2f} {r['safety']:>10.2f} {r['groundedness']:>10.2f}")

avg_rel = sum(r['relevance'] for r in eval_results) / len(eval_results)
avg_safe = sum(r['safety'] for r in eval_results) / len(eval_results)
avg_grnd = sum(r['groundedness'] for r in eval_results) / len(eval_results)
print("-" * 72)
print(f"  {'AVERAGE':<33} {avg_rel:>10.2f} {avg_safe:>10.2f} {avg_grnd:>10.2f}")

print(f"\n{'=' * 72}")
print("evaluate_agent.py (simplified):")
print(eval_code)
print("CONNECT TO PRIOR LABS:")
print("  Lab 06-01: MLflow evaluate() with built-in scorers")
print("  Lab 06-02: Custom scorers and Guidelines")
print("  Same scorers work here -- evaluation is consistent across the platform.")

**What just happened:** You saw how agent evaluation works in the template.
It uses the same `mlflow.genai.evaluate()` and scorers from Labs 06-01/06-02.
This consistency is intentional -- evaluate the same way locally and in production.

---

## Step 9: Framework Comparison -- Which OSS Library to Choose?

"Code Your Own Agent" supports multiple frameworks. The exam may ask when
to choose one over another.

In [ ]:
# ------------------------------------------------------------------
# Step 9: Framework comparison
# ------------------------------------------------------------------

frameworks = [
    {
        "name": "OpenAI Agents SDK",
        "strengths": "Simple tool-calling, streaming, easy to learn",
        "best_for": "Most agent use cases -- tool-calling chatbots, Q&A",
        "complexity": "Low",
        "key_concept": "Agent() + @function_tool",
        "template": "agent-openai-agents-sdk",
    },
    {
        "name": "LangGraph",
        "strengths": "Complex control flow, cycles, conditional routing",
        "best_for": "Multi-step workflows, human-in-the-loop, retries",
        "complexity": "Medium-High",
        "key_concept": "StateGraph with nodes and edges",
        "template": "agent-langgraph",
    },
    {
        "name": "DSPy",
        "strengths": "Programmatic prompt optimization, few-shot learning",
        "best_for": "When you need to optimize prompts automatically",
        "complexity": "Medium",
        "key_concept": "Modules + Optimizers + Metrics",
        "template": "agent-dspy",
    },
    {
        "name": "LangChain (LCEL)",
        "strengths": "Large ecosystem, many integrations, LCEL pipe syntax",
        "best_for": "RAG chains, standard pipelines, Databricks integrations",
        "complexity": "Low-Medium",
        "key_concept": "prompt | llm | parser (LCEL)",
        "template": "agent-langchain",
    },
]

print("=" * 72)
print("FRAMEWORK COMPARISON -- Code Your Own Agent")
print("=" * 72)

# Header
print(f"\n{'Framework':<22} {'Complexity':<14} {'Best For'}")
print("-" * 72)
for fw in frameworks:
    print(f"  {fw['name']:<20} {fw['complexity']:<14} {fw['best_for']}")

print(f"\n{'=' * 72}")
print("DECISION GUIDE:")
print("  Need a simple tool-calling agent?     -> OpenAI Agents SDK")
print("  Need complex control flow / cycles?    -> LangGraph")
print("  Need automatic prompt optimization?    -> DSPy")
print("  Need standard RAG / LCEL pipelines?    -> LangChain")
print("  Not sure?                              -> Start with OpenAI Agents SDK")
print()
print("ALL frameworks deploy the same way: agent.py + AgentServer + databricks apps deploy")

**What just happened:** You compared four OSS frameworks available for "Code Your Own Agent".
The key insight: **all deploy the same way** via AgentServer. The choice of framework
only affects your `agent.py` -- everything else (server, chat UI, auth, evaluation) is identical.

**Exam tip:** If the question mentions complex control flow or cycles, the answer is
**LangGraph**. If it mentions prompt optimization, it's **DSPy**. For everything else,
**OpenAI Agents SDK** or **LangChain** both work.

---

## Exam Tips & Traps

| # | Topic | Tip |
|---|---|---|
| 1 | When to use Code Your Own Agent | When pre-built bricks (Knowledge Assistant, Genie, etc.) don't fit your requirements and you need full control |
| 2 | MLflow AgentServer | Provides built-in REST API + chat UI automatically -- you don't build the server from scratch |
| 3 | Deployment command | `databricks apps deploy` (NOT `databricks deploy` or `databricks model serve`) |
| 4 | Authentication | Apps use **service principal** by default (app authorization). PATs are NOT supported |
| 5 | Framework choice | All frameworks deploy the same way. OpenAI Agents SDK is simplest; LangGraph for complex control flow |
| 6 | Evaluation | Uses the same `mlflow.genai.evaluate()` and scorers from Labs 06-01/06-02 |
| 7 | Local dev | `uv run start-app` starts the agent locally at localhost:8000 with full chat UI |

---

## Key Takeaways

### Core Concepts
- **Code Your Own Agent** = full OSS control over agent logic + Databricks handles infrastructure
- **Agent templates** provide agent code, server, chat UI, and evaluation out of the box
- **MLflow AgentServer** wraps your agent with REST API, chat UI, auth, and tracing
- You only modify **agent.py** (agent logic) and **app.yaml** (deployment config)
- All frameworks (OpenAI Agents SDK, LangGraph, DSPy, LangChain) deploy identically

### Databricks-Specific
- **UI ->** Agents -> Agent Bricks -> New -> Code Your Own Agent
- Deploy with CLI: `databricks apps create` -> `databricks sync` -> `databricks apps deploy`
- Apps use **app authorization** (auto-created service principal) by default
- Grant resources via **UI ->** Apps -> Edit -> Configure -> App resources
- PATs are NOT supported -- use OAuth tokens (`databricks auth token`)

### Exam Essentials
- "Code Your Own Agent" is for when pre-built bricks don't fit your needs
- You don't build the server from scratch -- AgentServer handles that
- Evaluation uses the same MLflow scorers across all deployment methods
- Local dev: `uv run quickstart` -> `uv run start-app` -> iterate -> deploy

---

## Next Lab

You have completed **Module 04: Assembling and Deploying Applications**.
Continue to **Module 05: Governance** (`05_governance/01_pii_masking_guardrails.ipynb`),
where you will learn about PII masking, data licensing, and governance controls
for GenAI applications.